In [204]:
from langgraph.graph import StateGraph , START , END
from langchain_mistralai import ChatMistralAI
from dotenv import load_dotenv
from typing_extensions import TypedDict , Annotated
from pydantic import BaseModel , Field
import operator
from rich import print

In [205]:
load_dotenv()

True

In [206]:
model = ChatMistralAI(model = "mistral-small-2506")

In [207]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback for the essay")
    score: int = Field(description="score out of 10" , ge=0 , le=10)

In [208]:
structured_model = model.with_structured_output(EvaluationSchema)

In [209]:
essay = """Artificial Intelligence (AI) has become one of the most transformative technologies of the 21st century. It is changing the way people live, work, learn, and interact with technology. In India, AI is playing a significant role in driving economic growth, improving public services, and fostering innovation across various sectors. As the country moves toward becoming a global digital leader, the responsible development and governance of AI have become equally important. Therefore, establishing clear rules and ethical guidelines for AI is essential to ensure that technology benefits society while minimizing its risks.

India has recognized the immense potential of AI in sectors such as healthcare, agriculture, education, finance, manufacturing, and governance. AI-powered tools are helping doctors detect diseases early, assisting farmers with crop prediction, enabling personalized education, and improving financial services through fraud detection and automation. Government initiatives like the National Strategy for Artificial Intelligence and the IndiaAI Mission aim to promote AI research, innovation, and skill development while encouraging startups and industries to adopt AI-driven solutions.

However, along with these opportunities come several challenges. AI systems often rely on large amounts of data, raising concerns about privacy, cybersecurity, and data protection. Biased datasets can result in unfair or discriminatory decisions in areas such as hiring, lending, or law enforcement. Additionally, the rapid automation of tasks may affect employment in certain industries, making reskilling and upskilling of the workforce essential. The spread of AI-generated misinformation and deepfake content also poses threats to democracy and public trust.

To address these concerns, India is focusing on creating a balanced regulatory framework that promotes innovation while ensuring accountability. The government emphasizes principles such as transparency, fairness, safety, inclusiveness, and respect for fundamental rights. AI developers and organizations are expected to build systems that are explainable, secure, and free from harmful biases. Human oversight should remain an essential part of AI decision-making, particularly in critical sectors like healthcare, finance, education, and criminal justice.

The Digital Personal Data Protection Act, 2023, strengthens the protection of personal data and provides an important foundation for responsible AI development. Additionally, various policy discussions encourage ethical AI practices, responsible data usage, and collaboration among government, academia, industry, and civil society. India also supports international cooperation to establish common standards for trustworthy AI while ensuring that regulations are suitable for the country's unique social and economic conditions.

Education and awareness play a crucial role in the successful adoption of AI. Schools, colleges, and training institutions should equip students with AI literacy, ethical understanding, and practical skills. Continuous learning programs can help workers adapt to technological changes and prepare for future job opportunities created by AI.

In conclusion, Artificial Intelligence has the potential to transform India's economy and improve the quality of life for millions of people. However, its success depends on responsible governance, ethical implementation, and effective regulation. By creating transparent rules, protecting citizens' rights, encouraging innovation, and investing in education and research, India can harness the full potential of AI while ensuring that technology serves humanity in a fair, inclusive, and sustainable manner. A balanced approach to AI regulation will enable India to emerge as a global leader in responsible Artificial Intelligence."""

In [210]:
prompt = f"Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10: \n {essay}"
structured_model.invoke(prompt)

EvaluationSchema(feedback='The essay is well-structured and provides a comprehensive overview of the role and impact of Artificial Intelligence in India. The language is clear and coherent, with a good flow of ideas. The use of technical terms is appropriate and well-explained, making the content accessible to a broad audience. The essay effectively highlights both the opportunities and challenges associated with AI, and it offers a balanced perspective on the need for responsible governance and ethical guidelines. The inclusion of specific examples, such as the Digital Personal Data Protection Act, 2023, and various government initiatives, strengthens the argument and demonstrates a thorough understanding of the subject. The conclusion effectively summarizes the main points and reinforces the importance of a balanced approach to AI regulation. Minor improvements could be made by ensuring consistent use of terminology and refining some sentence structures for enhanced readability.', sc

In [211]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int] , operator.add]
    avg_score: float


In [212]:
def evaluate_language(state: UPSCState):

    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [213]:
def evaluate_analysis(state: UPSCState):
    
    prompt = f"Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10: \n {state['essay']}"
    output = structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback , 'individual_scores': [output.score]}


In [214]:
def evaluate_thought(state: UPSCState):
    
    prompt = f"Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10: \n {state['essay']}"
    output = structured_model.invoke(prompt)

    return {'clarity_feedback': output.feedback , 'individual_scores': [output.score]}


In [215]:
def final_evaluation(state: UPSCState):

    # summary feedback
    prompt = f"Based on the following feedback create a summerized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}"
    overall_feedback = model.invoke(prompt).content

    # avg calculate
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback': overall_feedback , 'avg_score': avg_score}


In [216]:
graph = StateGraph(UPSCState)

# Add nodes
graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_thought", evaluate_thought)
graph.add_node("final_evaluation", final_evaluation)

# Add edges
graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_analysis")
graph.add_edge(START, "evaluate_thought")

graph.add_edge("evaluate_language", "final_evaluation")
graph.add_edge("evaluate_analysis", "final_evaluation")
graph.add_edge("evaluate_thought", "final_evaluation")

graph.add_edge("final_evaluation", END)

workflow = graph.compile()
print(workflow)

<langgraph.graph.state.CompiledStateGraph object at 0x000001A0C5F25DF0>

In [217]:
intial_state = {
    'essay': essay
}

workflow.invoke(intial_state)

{'essay': "Artificial Intelligence (AI) has become one of the most transformative technologies of the 21st century. It is changing the way people live, work, learn, and interact with technology. In India, AI is playing a significant role in driving economic growth, improving public services, and fostering innovation across various sectors. As the country moves toward becoming a global digital leader, the responsible development and governance of AI have become equally important. Therefore, establishing clear rules and ethical guidelines for AI is essential to ensure that technology benefits society while minimizing its risks.\n\nIndia has recognized the immense potential of AI in sectors such as healthcare, agriculture, education, finance, manufacturing, and governance. AI-powered tools are helping doctors detect diseases early, assisting farmers with crop prediction, enabling personalized education, and improving financial services through fraud detection and automation. Government in